# Tavole e report CSV – Reti violenza

Questo notebook genera **tavole descrittive, controlli e grafici** per ciascun file CSV prodotto dal flusso di analisi.

## Cosa contiene
- caricamento automatico dei CSV
- tavola iniziale con dimensioni e colonne
- report dedicato per ogni file
- riepiloghi per regione
- totali delle colonne numeriche
- grafici base per i prospetti regionali
- controlli mirati per `tabella_reti.csv` e `tabella_soggetti.csv`

> Percorso atteso dei file: stessa cartella del notebook, oppure cartella specificata in `BASE_DIR`.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# ==================================================
# DIRECTORY
# ==================================================
BASE_DIR = Path("output/data/step_3.2")
PNG_DIR = Path("output/reports/png")
PNG_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# ==================================================
# LOAD CSV
# ==================================================
def must_load_csv(base_dir, filename):
    path = Path(base_dir) / filename
    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")
    print(f"Caricato: {path}")
    return pd.read_csv(path)

prospetto_reti_per_regione = must_load_csv(BASE_DIR, "prospetto_reti_per_regione.csv")
prospetto_ambito_per_regione = must_load_csv(BASE_DIR, "prospetto_ambito_per_regione.csv")
prospetto_soggetti_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_soggetti_tipo10_per_regione.csv")
prospetto_soggetti_tipo30_per_regione = must_load_csv(BASE_DIR, "prospetto_soggetti_tipo30_per_regione.csv")
prospetto_attori_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_attori_tipo10_per_regione.csv")
prospetto_proponenti_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_proponenti_tipo10_per_regione.csv")
prospetto_firmatari_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_firmatari_tipo10_per_regione.csv")
prospetto_governance_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_governance_tipo10_per_regione.csv")


tabella_reti = must_load_csv(BASE_DIR, "tabella_reti.csv")
tabella_soggetti = must_load_csv(BASE_DIR, "tabella_soggetti.csv")


loaded_tables = {
    "prospetti": {
        "Reti per Regione": prospetto_reti_per_regione,
        "Ambito per Regione": prospetto_ambito_per_regione,
        "Soggetti per Regione × Classe 10": prospetto_soggetti_tipo10_per_regione,
        "Soggetti per Regione × Classe 30": prospetto_soggetti_tipo30_per_regione,
        "Attori per Regione × Classe 10": prospetto_attori_tipo10_per_regione,
        "Proponenti per Regione × Classe 10": prospetto_proponenti_tipo10_per_regione,
        "Firmatari per Regione × Classe 10": prospetto_firmatari_tipo10_per_regione,        
        "Governance per Regione × Classe 10": prospetto_governance_tipo10_per_regione,     
    },
    "tabelle": {
        "Tabella Reti": tabella_reti,
        "Tabella Soggetti": tabella_soggetti,
    }
}

tabella_soggetti.head(), tabella_reti.head()


print("\nTabelle caricate:", len(loaded_tables))
print("Cartella PNG:", PNG_DIR.resolve())

Caricato: output\data\step_3.2\prospetto_reti_per_regione.csv
Caricato: output\data\step_3.2\prospetto_ambito_per_regione.csv
Caricato: output\data\step_3.2\prospetto_soggetti_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_soggetti_tipo30_per_regione.csv
Caricato: output\data\step_3.2\prospetto_attori_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_proponenti_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_firmatari_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_governance_tipo10_per_regione.csv
Caricato: output\data\step_3.2\tabella_reti.csv
Caricato: output\data\step_3.2\tabella_soggetti.csv

Tabelle caricate: 2
Cartella PNG: G:\develpment\protocolli-intesa\output\reports\png


In [4]:
overview = []

for sezione, blocco in loaded_tables.items():
    for name, df in blocco.items():
        overview.append({
            "sezione": sezione,
            "file": name,
            "righe": len(df),
            "colonne": len(df.columns),
            "colonne_numeriche": len(df.select_dtypes(include="number").columns),
            "ha_regione": "regione" in df.columns
        })

overview_df = (
    pd.DataFrame(overview)
    .sort_values(["sezione", "file"])
    .reset_index(drop=True)
)

overview_df


,sezione,file,righe,colonne,colonne_numeriche,ha_regione
0,prospetti,Ambito per Regione,17,9,8,True
1,prospetti,Attori per Regione × Classe 10,17,12,11,True
2,prospetti,Firmatari per Regione × Classe 10,15,12,11,True
3,prospetti,Governance per Regione × Classe 10,15,12,11,True
4,prospetti,Proponenti per Regione × Classe 10,15,12,11,True
5,prospetti,Reti per Regione,17,7,6,True
6,prospetti,Soggetti per Regione × Classe 10,17,12,11,True
7,prospetti,Soggetti per Regione × Classe 30,17,31,30,True
8,tabelle,Tabella Reti,240,16,9,True
9,tabelle,Tabella Soggetti,4502,19,7,True


## Funzioni di supporto

In [5]:
def numeric_columns(df: pd.DataFrame):
    return list(df.select_dtypes(include="number").columns)

def safe_display(df: pd.DataFrame, n: int = 10):
    if df.empty:
        print("DataFrame vuoto")
    else:
        display(df.head(n))

def report_table(df: pd.DataFrame, group_col: str, top_n: int = 20):
    if group_col not in df.columns:
        print(f"Colonna '{group_col}' non presente")
        return
    out = (
        df.groupby(group_col)
        .size()
        .reset_index(name="totale")
        .sort_values(["totale", group_col], ascending=[False, True])
        .head(top_n)
        .reset_index(drop=True)
    )
    display(out)

def show_numeric_totals(df: pd.DataFrame):
    nums = numeric_columns(df)
    if not nums:
        print("Nessuna colonna numerica")
        return
    totals = pd.DataFrame({
        "colonna": nums,
        "totale": [df[c].sum() for c in nums]
    }).sort_values("colonna").reset_index(drop=True)
    display(totals)




In [6]:
def read_csv_flexible(path: Path) -> pd.DataFrame:
    encodings = ['utf-8', 'utf-8-sig', 'latin1', 'cp1252']
    seps = [',', ';', '\t']
    last_error = None
    for enc in encodings:
        for sep in seps:
            try:
                return pd.read_csv(path, encoding=enc, sep=sep)
            except Exception as e:
                last_error = e
    raise RuntimeError(f'Impossibile leggere {path}: {last_error}')

def autosize_height(n_rows: int, base: float = 4.0, scale: float = 0.20, max_size: float = 16.0) -> float:
    return min(max_size, max(base, base + n_rows * scale))

def save_barh(df: pd.DataFrame, y_col: str, x_col: str, title: str, out_path: Path) -> None:
    if df.empty:
        print('skip', out_path.name, '(empty)')
        return
    plot_df = df.copy().sort_values(x_col, ascending=True)
    h = autosize_height(len(plot_df), base=4.0, scale=0.20, max_size=16.0)
    plt.figure(figsize=(12, h))
    plt.barh(plot_df[y_col].astype(str), plot_df[x_col])
    plt.title(title)
    plt.xlabel('Numero')
    plt.ylabel('')
    plt.tight_layout()
    plt.tick_params(axis='y', labelsize=9)
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()
    print('saved', out_path)

    



# Numero reti per regione

In [7]:
df = tabella_reti

if "regione" in df.columns:
    figura = (
        df.groupby("regione", dropna=False)
        .size()
        .reset_index(name="numero")
        .sort_values(["numero", "regione"], ascending=[False, True])
    )
else:
    figura1 = pd.DataFrame(columns=["regione", "numero"])

save_barh(
    figura,
    "regione",
    "numero",
    "Figura 1.1. Numero reti per regione",
    PNG_DIR / "1.1_figura_numero_reti_per_regione.png"
)

saved output\reports\png\1.1_figura_numero_reti_per_regione.png


# Numero soggetti per regione

In [8]:

df = tabella_soggetti

if "regione" in df.columns:
    figura = (
        df.groupby("regione", dropna=False)
        .size()
        .reset_index(name="numero")
        .sort_values(["numero", "regione"], ascending=[False, True])
    )
else:
    figura = pd.DataFrame(columns=["regione", "numero"])

save_barh(
    figura,
    "regione",
    "numero",
    "Figura 1.2. Numero soggetti per regione",
    PNG_DIR / "1.2_figura_numero_soggetti_per_regione.png"
)

saved output\reports\png\1.2_figura_numero_soggetti_per_regione.png


# Proponenti vs attori per regione

In [9]:
def save_figura_grouped(df: pd.DataFrame, out_path: Path) -> None:
    if df.empty:
        print('skip', out_path.name, '(empty)')
        return
    plot_df = df.copy()
    x = range(len(plot_df))
    width = 0.42
    plt.figure(figsize=(12, autosize_height(len(plot_df), base=4.0, scale=0.15, max_size=10.0)))
    plt.barh([i - width / 2 for i in x], plot_df['soggetti_proponenti'], height=width, label='Soggetti proponenti')
    plt.barh([i + width / 2 for i in x], plot_df['attori_coinvolti'], height=width, label='Attori coinvolti')
    plt.yticks(list(x), plot_df['regione'].astype(str))
    plt.title('Figura 4. Soggetti proponenti e attori coinvolti per regione')
    plt.xlabel('Numero')
    plt.ylabel('')
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()
    print('saved', out_path)



if {'ruolo_proponente', 'ruolo_attore', 'regione', 'nome_soggetto'}.issubset(tabella_soggetti.columns):
    prop = (tabella_soggetti.loc[tabella_soggetti['ruolo_proponente'] == 1]
            .groupby('regione', dropna=False)
            .agg(soggetti_proponenti=('nome_soggetto', 'size'))
            .reset_index())
    att = (tabella_soggetti.loc[tabella_soggetti['ruolo_attore'] == 1]
           .groupby('regione', dropna=False)
           .agg(attori_coinvolti=('nome_soggetto', 'size'))
           .reset_index())
    figura = prop.merge(att, on='regione', how='outer').fillna(0)
    for c in ['soggetti_proponenti', 'attori_coinvolti']:
        figura[c] = figura[c].astype(int)
else:
    figura = pd.DataFrame(columns=['regione', 'soggetti_proponenti', 'attori_coinvolti'])

save_figura_grouped(figura, PNG_DIR / '1.3_figura_proponenti_vs_attori_per_regione.png')

saved output\reports\png\1.3_figura_proponenti_vs_attori_per_regione.png


# Attori coinvolti per tipo 10

In [10]:

if {'ruolo_attore', 'tipo_aggregato_10', 'nome_soggetto'}.issubset(tabella_soggetti.columns):
    figura = (tabella_soggetti.loc[tabella_soggetti['ruolo_attore'] == 1]
               .groupby('tipo_aggregato_10', dropna=False)
               .agg(numero=('nome_soggetto', 'size'))
               .reset_index()
               .rename(columns={'tipo_aggregato_10': 'tipologia'})
               .sort_values(['numero', 'tipologia'], ascending=[False, True]))
else:
    figura = pd.DataFrame(columns=['tipologia', 'numero'])

save_barh(figura, 'tipologia', 
          'numero', 
          'Figura 2.1. Attori coinvolti per aggregazione 10',
            PNG_DIR / '2.1_figura_attori_coinvolti_per_tipo10.png')

saved output\reports\png\2.1_figura_attori_coinvolti_per_tipo10.png


# Ambiti territoriali

# Soggetti per classe 30 (Top 10)

In [11]:

df = prospetto_soggetti_tipo30_per_regione

if "regione" in df.columns:
    value_cols = [c for c in df.columns if c != "regione"]
else:
    value_cols = list(df.columns)

if value_cols:
    figura = (
        df[value_cols]
        .sum(axis=0)
        .reset_index()
    )
    figura.columns = ["classe_30", "numero"]

    figura = (
        figura.sort_values(
            ["numero", "classe_30"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )
else:
    figura = pd.DataFrame(columns=["classe_30", "numero"])

save_barh(
    figura,
    "classe_30",
    "numero",
    "Figura 2.2 Soggetti per classe 30",
    PNG_DIR / "2.2_figura_soggetti_per_classe30.png"
)

saved output\reports\png\2.2_figura_soggetti_per_classe30.png


# Soggetti proponenti per tipo 10

In [12]:

df = tabella_soggetti

if {'ruolo_proponente', 'tipo_aggregato_10', 'nome_soggetto'}.issubset(df.columns):
    figura = (
        df.loc[df['ruolo_proponente'] == 1]
        .groupby('tipo_aggregato_10', dropna=False)
        .agg(numero=('nome_soggetto', 'size'))
        .reset_index()
        .rename(columns={'tipo_aggregato_10': 'tipologia'})
        .sort_values(['numero', 'tipologia'], ascending=[False, True])
    )
else:
    figura = pd.DataFrame(columns=['tipologia', 'numero'])

save_barh(
    figura,
    'tipologia',
    'numero',
    'Figura 3.1 Soggetti proponenti per aggregazione 10',
    PNG_DIR / '3.1_figura_soggetti_proponenti_per_aggregazione_10.png'
)

saved output\reports\png\3.1_figura_soggetti_proponenti_per_aggregazione_10.png


# Soggetti proponenti per regione

In [13]:

if {'ruolo_proponente', 'regione', 'nome_soggetto'}.issubset(tabella_soggetti.columns):
    figura = (
        tabella_soggetti.loc[tabella_soggetti['ruolo_proponente'] == 1]
               .groupby('regione', dropna=False)
               .agg(numero=('nome_soggetto', 'size'))
               .reset_index()
               )
else:
    figura = pd.DataFrame(columns=['regione', 'numero'])

save_barh(figura, 
          'regione', 
          'numero', 
          'Figura 3.2 Soggetti proponenti per regione', 
          PNG_DIR / '3.2_figura_soggetti_proponenti_per_regione.png')

saved output\reports\png\3.2_figura_soggetti_proponenti_per_regione.png


# Firmatari per classe 10

In [14]:

df = prospetto_firmatari_tipo10_per_regione

if "regione" in df.columns:
    value_cols = [c for c in df.columns if c != "regione"]
else:
    value_cols = list(df.columns)

if value_cols:
    figura = (
        df[value_cols]
        .sum(axis=0)
        .reset_index()
    )
    figura .columns = ["aggregazione_10", "numero"]
    figura = figura.sort_values(
        ["numero", "aggregazione_10"],
        ascending=[False, True]
    ).reset_index(drop=True)
else:
    figura = pd.DataFrame(columns=["aggregazione_10", "numero"])

save_barh(
    figura,
    "aggregazione_10",
    "numero",
    "Figura 3.3. Firmatari per classe 10",
    PNG_DIR / "3.3_figura_firmatari_per_aggregazione_10.png"
)

saved output\reports\png\3.3_figura_firmatari_per_aggregazione_10.png


# Governance per classe 10

In [15]:

df = prospetto_governance_tipo10_per_regione

if "regione" in df.columns:
    value_cols = [c for c in df.columns if c != "regione"]
else:
    value_cols = list(df.columns)

if value_cols:
    figura = (
        df[value_cols]
        .sum(axis=0)
        .reset_index()
    )
    figura.columns = ["aggregazione_10", "numero"]
    figura = figura.sort_values(["numero", "aggregazione_10"], ascending=[False, True]).reset_index(drop=True)
else:
    figura = pd.DataFrame(columns=["aggregazione_10", "numero"])

save_barh(
    figura,
    "aggregazione_10",
    "numero",
    "Figura 3.4. Governance per aggregazione 10",
    PNG_DIR / "3.4_figura_governance_per_aggregazione_10.png"
)

saved output\reports\png\3.4_figura_governance_per_aggregazione_10.png


# Governance per regione

In [16]:

df = prospetto_governance_tipo10_per_regione

if "regione" in df.columns:
    value_cols = [c for c in df.columns if c != "regione"]

    figura = (
        df.assign(numero=df[value_cols].sum(axis=1))
        [["regione", "numero"]]
        .sort_values(["numero", "regione"], ascending=[False, True])
        .reset_index(drop=True)
    )
else:
    figura10 = pd.DataFrame(columns=["regione", "numero"])

save_barh(
    figura,
    "regione",
    "numero",
    "Figura 3.5. Governance per regione",
    PNG_DIR / "3.5_figura_governance_per_regione.png"
)

saved output\reports\png\3.5_figura_governance_per_regione.png


# Ambito per regione

In [17]:

df = prospetto_ambito_per_regione

if "regione" in df.columns:
    value_cols = [c for c in df.columns if c != "regione"]

    figura = (
        df.assign(numero=df[value_cols].sum(axis=1))
        [["regione", "numero"]]
        .sort_values(["numero", "regione"], ascending=[False, True])
        .reset_index(drop=True)
    )
else:
    figura11 = pd.DataFrame(columns=["regione", "numero"])

save_barh(
    figura,
    "regione",
    "numero",
    "Figura 4.1. Ambito per regione",
    PNG_DIR / "4.1_figura_ambito_per_regione.png"
)

saved output\reports\png\4.1_figura_ambito_per_regione.png


# Ambiti territoriali delle reti

In [18]:

df = loaded_tables["tabelle"]["Tabella Reti"]

if "ambito_territoriale" in df.columns:
    figura = (
        df.groupby("ambito_territoriale", dropna=False)
        .size()
        .reset_index(name="numero")
        .sort_values(
            ["numero", "ambito_territoriale"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )
else:
    figura = pd.DataFrame(columns=["ambito_territoriale", "numero"])

save_barh(
    figura,
    "ambito_territoriale",
    "numero",
    "Figura 4.2. Ambiti territoriali delle reti",
    PNG_DIR / "4.2_figura_ambiti_territoriali.png"
)

saved output\reports\png\4.2_figura_ambiti_territoriali.png


In [19]:
readme = [
    'REPORT GRAFICI GENERATI',
    '======================',
    '',
    'Le figure sono organizzate per area tematica.',
    '',
    '# SEZIONE A — QUADRO GENERALE',
    '- 1.1_figura_numero_reti_per_regione.png -> Numero di reti censite per regione',
    '- 1.2_figura_numero_soggetti_per_regione.png -> Totale soggetti coinvolti per regione',
    '- 1.3_figura_proponenti_vs_attori_per_regione.png -> Confronto tra soggetti proponenti e attori coinvolti per regione',
    '',
    '# SEZIONE B — COMPOSIZIONE DELLA RETE',
    '- 2.1_figura_attori_coinvolti_per_tipo10.png -> Attori coinvolti per macro-tipologia (aggregazione 10)',
    '- 2.3_figura_soggetti_per_classe30.png -> Distribuzione soggetti per classificazione dettagliata (classe 30)',
    '',
    '# SEZIONE C — GOVERNANCE, FIRME E INIZIATIVA',
    '- 3.1_figura_soggetti_proponenti_per_aggregazione_10.png -> Soggetti promotori per aggregazione 10',
    '- 3.2_figura_soggetti_proponenti_per_regione.png -> Soggetti promotori per regione',
    '- 3.3_figura_firmatari_per_aggregazione_10.png -> Firmatari per aggregazione 10',
    '- 3.4_figura_governance_per_aggregazione_10.png -> Soggetti con ruolo di governance per aggregazione 10',
    '- 3.5_figura_governance_per_regione.png -> Soggetti con ruolo di governance per regione',
    '',
    '# SEZIONE D — DIMENSIONE TERRITORIALE',
    
    '- 4.1_figura_ambito_per_regione.png -> Ambito territoriale prevalente per regione',
    '- 4.2_figura_ambiti_territoriali.png -> Riepilogo finale degli ambiti territoriali delle reti',

    '',
    'NOTE',
    '----',
    '- I grafici numerati 1.x descrivono il quadro generale.',
    '- I grafici numerati 2.x descrivono la composizione interna delle reti.',
    '- I grafici numerati 3.x analizzano promotori, firmatari e governance.',
    '- I grafici numerati 4.x riguardano la distribuzione territoriale.',
]

(PNG_DIR / "README_png.txt").write_text("\n".join(readme), encoding="utf-8")
print("Creato:", PNG_DIR / "README_png.txt")

Creato: output\reports\png\README_png.txt


In [20]:
from pathlib import Path
import pandas as pd

HTML_DIR = Path("output/reports/html")
HTML_DIR.mkdir(parents=True, exist_ok=True)


def make_html_table_with_totals(
    df: pd.DataFrame,
    row_label: str = "regione",
    row_total_label: str = "Totale riga",
    col_total_label: str = "Totale colonna",
) -> pd.DataFrame:
    out = df.copy()

    if row_label not in out.columns:
        raise ValueError(f"Colonna '{row_label}' non presente")

    value_cols = [c for c in out.columns if c != row_label]

    # totale riga
    out[row_total_label] = out[value_cols].sum(axis=1)

    # riga totale colonna
    total_row = {row_label: col_total_label}
    for c in value_cols:
        total_row[c] = out[c].sum()
    total_row[row_total_label] = out[row_total_label].sum()

    out = pd.concat([out, pd.DataFrame([total_row])], ignore_index=True)
    return out


def dataframe_to_html_table(
    df: pd.DataFrame,
    title: str,
    out_path: Path,
    row_label: str = "regione"
) -> None:
    table_df = make_html_table_with_totals(df, row_label=row_label)

    html = f"""
<!DOCTYPE html>
<html lang="it">
<head>
<meta charset="utf-8">
<title>{title}</title>
<style>
body {{
    font-family: Arial, Helvetica, sans-serif;
    margin: 24px;
    color: #222;
}}
h1 {{
    font-size: 22px;
    margin-bottom: 16px;
}}
table {{
    border-collapse: collapse;
    width: 100%;
    font-size: 14px;
}}
th, td {{
    border: 1px solid #999;
    padding: 6px 8px;
}}
th {{
    background: #f0f0f0;
    text-align: center;
}}
td:first-child {{
    font-weight: 600;
}}
td {{
    text-align: right;
}}
td:first-child {{
    text-align: left;
}}
tr:last-child {{
    background: #f7f7f7;
    font-weight: 700;
}}
</style>
</head>
<body>
<h1>{title}</h1>
{table_df.to_html(index=False, border=0, escape=False)}
</body>
</html>
""".strip()

    out_path.write_text(html, encoding="utf-8")
    print("Creato:", out_path)

In [21]:
html_tables = {
    "1.1_tavola_reti_per_regione.html": loaded_tables["prospetti"]["Reti per Regione"],
    "2.1_tavola_attori_per_aggregazione_10.html": loaded_tables["prospetti"]["Attori per Regione × Classe 10"],
    "2.2_tavola_soggetti_per_aggregazione_10.html": loaded_tables["prospetti"]["Soggetti per Regione × Classe 10"],
    "2.3_tavola_soggetti_per_classe30.html": loaded_tables["prospetti"]["Soggetti per Regione × Classe 30"],
    "3.1_tavola_proponenti_per_aggregazione_10.html": loaded_tables["prospetti"]["Proponenti per Regione × Classe 10"],
    "3.2_tavola_governance_per_aggregazione_10.html": loaded_tables["prospetti"]["Governance per Regione × Classe 10"],
    "4.1_tavola_ambito_per_regione.html": loaded_tables["prospetti"]["Ambito per Regione"],
}

for filename, df in html_tables.items():
    title = filename.replace(".html", "").replace("_", " ")
    dataframe_to_html_table(
        df,
        title=title,
        out_path=HTML_DIR / filename,
        row_label="regione"
    )

Creato: output\reports\html\1.1_tavola_reti_per_regione.html
Creato: output\reports\html\2.1_tavola_attori_per_aggregazione_10.html
Creato: output\reports\html\2.2_tavola_soggetti_per_aggregazione_10.html
Creato: output\reports\html\2.3_tavola_soggetti_per_classe30.html
Creato: output\reports\html\3.1_tavola_proponenti_per_aggregazione_10.html
Creato: output\reports\html\3.2_tavola_governance_per_aggregazione_10.html
Creato: output\reports\html\4.1_tavola_ambito_per_regione.html
